In [ ]:
import os, time, math, random, copy, json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE, "| torch:", torch.__version__, "| torchvision:", torchvision.__version__)

In [ ]:
SMOKE_TEST = False
USE_WANDB = True
WANDB_PROJECT = "simclr-cifar10-ablations"
WANDB_ENTITY = None  # optionally set your W&B team/entity
DATA_ROOT = "./dataset"
OUTPUT_DIR = Path("./simclr_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_CONFIG = {
    "seed": SEED,
    "dataset": "CIFAR10",
    "backbone": "ResNet18",   # retained from the provided notebook
    "epochs": 1 if SMOKE_TEST else 100,
    "batch_size": 64 if SMOKE_TEST else 256,
    "num_workers": 2,
    "learning_rate": 0.3,     # SimCLR-style SGD learning rate, scaled with batch size below
    "weight_decay": 1e-4,
    "temperature": 0.5,
    "projection_dim": 128,
    "projection_hidden_dim": 512,
    "use_projection_head": True,
    "use_color_distortion": True,
    "use_blur": True,
    "use_random_crop": True,
    "use_horizontal_flip": True,
    "subset_size": 2048 if SMOKE_TEST else None,
    "log_every_steps": 50,
}

In [ ]:
def make_view_transform(cfg):
    ops = []
    if cfg["use_random_crop"]:
        ops.append(transforms.RandomResizedCrop(32, scale=(0.08, 1.0), interpolation=transforms.InterpolationMode.BICUBIC))
    else:
        ops.append(transforms.Resize((32, 32)))
    if cfg["use_horizontal_flip"]:
        ops.append(transforms.RandomHorizontalFlip(p=0.5))
    if cfg["use_color_distortion"]:
        jitter = transforms.ColorJitter(0.8, 0.8, 0.8, 0.2)
        ops.extend([transforms.RandomApply([jitter], p=0.8), transforms.RandomGrayscale(p=0.2)])
    if cfg["use_blur"]:
        ops.append(transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.5))
    ops.extend([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ])
    return transforms.Compose(ops)

class TwoViewTransform:
    def __init__(self, transform):
        self.transform = transform
    def __call__(self, image):
        return self.transform(image), self.transform(image)

def make_loaders(cfg):
    transform = TwoViewTransform(make_view_transform(cfg))
    train_ds = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=transform)
    if cfg.get("subset_size"):
        # deterministic subset for smoke tests
        g = torch.Generator().manual_seed(cfg["seed"])
        indices = torch.randperm(len(train_ds), generator=g)[:cfg["subset_size"]].tolist()
        train_ds = torch.utils.data.Subset(train_ds, indices)
    loader = DataLoader(
        train_ds, batch_size=cfg["batch_size"], shuffle=True, drop_last=True,
        num_workers=cfg["num_workers"], pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(cfg["num_workers"] > 0)
    )
    return loader

In [ ]:
class SimCLR(nn.Module):
    def __init__(self, projection_hidden_dim=512, projection_dim=128, use_projection_head=True):
        super().__init__()
        backbone = models.resnet18(weights=None)
        feature_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.encoder = backbone
        self.use_projection_head = use_projection_head
        self.projector = nn.Sequential(
            nn.Linear(feature_dim, projection_hidden_dim, bias=False),
            nn.BatchNorm1d(projection_hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(projection_hidden_dim, projection_dim, bias=True),
        ) if use_projection_head else nn.Identity()
        self.representation_dim = feature_dim
        self.projection_dim = projection_dim if use_projection_head else feature_dim

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return h, z

def nt_xent_loss(z1, z2, temperature=0.5):
    # z1,z2: [N,D]. Each view's matching image is the positive; all other views are negatives.
    n = z1.shape[0]
    z = F.normalize(torch.cat([z1, z2], dim=0), dim=1)
    logits = (z @ z.T) / temperature
    logits.fill_diagonal_(float("-inf"))
    targets = torch.cat([
        torch.arange(n, 2*n, device=z.device),
        torch.arange(0, n, device=z.device)
    ])
    return F.cross_entropy(logits, targets)



In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def train_one_run(cfg, run_name=None, use_wandb=True):
    cfg = dict(cfg)
    set_seed(cfg["seed"])
    loader = make_loaders(cfg)
    model = SimCLR(
        projection_hidden_dim=cfg["projection_hidden_dim"],
        projection_dim=cfg["projection_dim"],
        use_projection_head=cfg["use_projection_head"],
    ).to(DEVICE)

    lr = cfg["learning_rate"] * cfg["batch_size"] / 256.0
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"])
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
    wb_run = None
    should_log = bool(use_wandb and USE_WANDB and WANDB_AVAILABLE)
    if should_log:
        wb_run = wandb.init(
            project=WANDB_PROJECT, entity=WANDB_ENTITY, name=run_name,
            config={**cfg, "effective_learning_rate": lr},
            reinit=True, job_type="simclr-ablation"
        )
        wandb.watch(model, log="gradients", log_freq=max(100, cfg["log_every_steps"]))

    history = []
    global_step = 0
    for epoch in range(cfg["epochs"]):
        model.train()
        epoch_loss, seen, epoch_start = 0.0, 0, time.time()
        progress = tqdm(loader, desc=f"{run_name or 'SimCLR'} | epoch {epoch+1}/{cfg['epochs']}", leave=False)
        for (x1, x2), _ in progress:
            x1, x2 = x1.to(DEVICE, non_blocking=True), x2.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                _, z1 = model(x1)
                _, z2 = model(x2)
                loss = nt_xent_loss(z1, z2, cfg["temperature"])
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            bs = x1.size(0)
            epoch_loss += loss.item() * bs
            seen += bs
            global_step += 1
            progress.set_postfix(loss=f"{loss.item():.3f}")
            if should_log and global_step % cfg["log_every_steps"] == 0:
                wandb.log({"train/step_loss": loss.item(), "train/lr": optimizer.param_groups[0]["lr"], "global_step": global_step}, step=global_step)

        scheduler.step()
        elapsed = time.time() - epoch_start
        row = {
            "epoch": epoch + 1,
            "train_loss": epoch_loss / max(seen, 1),
            "learning_rate": optimizer.param_groups[0]["lr"],
            "epoch_seconds": elapsed,
            "images_per_second": seen / max(elapsed, 1e-9),
            "examples_seen": seen,
        }
        history.append(row)
        if should_log:
            wandb.log({f"epoch/{k}": v for k, v in row.items()})
        print(f"epoch {epoch+1:03d}/{cfg['epochs']} | loss={row['train_loss']:.4f} | {row['images_per_second']:.1f} images/s")

    # Save encoder + projector + optimizer state for reproducibility
    safe_name = (run_name or "simclr").replace("/", "_").replace(" ", "_")
    ckpt_path = OUTPUT_DIR / f"{safe_name}.pt"
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": cfg,
        "history": history,
    }, ckpt_path)
    pd.DataFrame(history).to_csv(OUTPUT_DIR / f"{safe_name}_history.csv", index=False)
    if should_log:
        wandb.log({"checkpoint_path": str(ckpt_path)})
        wandb.finish()
    return model, pd.DataFrame(history), str(ckpt_path)

In [ ]:
ABLATIONS = [
    ("baseline", {}),
    ("no_color_distortion", {"use_color_distortion": False}),
    ("no_gaussian_blur", {"use_blur": False}),
    ("no_projection_head", {"use_projection_head": False}),
    ("temperature_0.1", {"temperature": 0.1}),
    ("temperature_1.0", {"temperature": 1.0}),
]


print("Runs:", [name for name, _ in ABLATIONS])

In [ ]:
all_histories = {}
all_configs = {}
trained_models = {}
checkpoint_paths = {}

for run_name, overrides in ABLATIONS:
    cfg = {**BASE_CONFIG, **overrides}
    print("\n" + "="*80 + "\nStarting:", run_name, "| overrides:", overrides)
    model, hist, ckpt = train_one_run(cfg, run_name=run_name, use_wandb=USE_WANDB)
    all_histories[run_name] = hist.assign(run=run_name)
    all_configs[run_name] = cfg
    checkpoint_paths[run_name] = ckpt
    # keep only the baseline model in memory to reduce RAM/VRAM usage
    if run_name == "baseline":
        trained_models[run_name] = model
    else:
        del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

results_df = pd.concat(all_histories.values(), ignore_index=True)
results_df.to_csv(OUTPUT_DIR / "all_ablation_histories.csv", index=False)
results_df.head()

In [ ]:
# Training loss curves
fig, ax = plt.subplots(figsize=(10, 6))
for name, hist in all_histories.items():
    ax.plot(hist["epoch"], hist["train_loss"], label=name)
ax.set(title="SimCLR training loss by ablation", xlabel="Epoch", ylabel="NT-Xent loss")
ax.grid(True, alpha=0.3); ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "training_loss_ablation.png", dpi=160)
plt.show()

# Mean final loss, speed, and total runtime
summary_rows = []
for name, hist in all_histories.items():
    summary_rows.append({
        "run": name,
        "final_train_loss": float(hist["train_loss"].iloc[-1]),
        "mean_images_per_second": float(hist["images_per_second"].mean()),
        "total_training_seconds": float(hist["epoch_seconds"].sum()),
        "epochs": len(hist),
    })
summary_df = pd.DataFrame(summary_rows).sort_values("final_train_loss")
summary_df.to_csv(OUTPUT_DIR / "ablation_summary.csv", index=False)
display(summary_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(summary_df["run"], summary_df["final_train_loss"])
axes[0].set_title("Final training loss"); axes[0].set_ylabel("NT-Xent loss")
axes[1].bar(summary_df["run"], summary_df["mean_images_per_second"])
axes[1].set_title("Throughput"); axes[1].set_ylabel("Images / second")
for ax in axes:
    ax.tick_params(axis="x", rotation=35); ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "ablation_summary.png", dpi=160)
plt.show()

In [ ]:
# Log aggregate plots + ablation table to W&B (in addition to per-run training metrics)
if USE_WANDB and WANDB_AVAILABLE:
    with wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="ablation-summary", job_type="summary") as summary_run:
        summary_run.log({
            "plots/training_loss_ablation": wandb.Image(str(OUTPUT_DIR / "training_loss_ablation.png")),
            "plots/ablation_summary": wandb.Image(str(OUTPUT_DIR / "ablation_summary.png")),
            "plots/two_augmented_views": wandb.Image(str(OUTPUT_DIR / "two_augmented_views.png")) if (OUTPUT_DIR / "two_augmented_views.png").exists() else None,
            "ablation/table": wandb.Table(dataframe=summary_df),
        })
        summary_run.summary["best_final_train_loss_run"] = str(summary_df.iloc[0]["run"])
        summary_run.summary["runs_completed"] = int(len(summary_df))
else:
    print("W&B aggregate logging skipped (disabled or wandb is not installed).")


In [ ]:
def show_two_views(index=0, cfg=None):
    cfg = cfg or BASE_CONFIG
    raw_ds = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=None)
    image, label = raw_ds[index]
    tfm = make_view_transform(cfg)
    a, b = tfm(image), tfm(image)
    mean = torch.tensor((0.4914, 0.4822, 0.4465)).view(3,1,1)
    std = torch.tensor((0.2470, 0.2435, 0.2616)).view(3,1,1)
    def unnormalize(x):
        return (x * std + mean).clamp(0,1).permute(1,2,0).numpy()
    fig, axes = plt.subplots(1, 2, figsize=(6,3))
    axes[0].imshow(unnormalize(a)); axes[0].set_title("View 1")
    axes[1].imshow(unnormalize(b)); axes[1].set_title("View 2")
    for ax in axes: ax.axis("off")
    fig.suptitle(f"CIFAR-10 label: {raw_ds.classes[label]}")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "two_augmented_views.png", dpi=160)
    plt.show()

show_two_views()